# AutoPBR v2 — Run Pipeline (config-driven)

Questo notebook esegue la pipeline leggendo **un unico file di configurazione**:

- `config/pipeline.yaml`

Vantaggi:
- un solo posto dove cambiare parametri (SAM3 / Nemotron / soglie / paths)
- riproducibile per colleghi e reviewer
- stesso comportamento del terminale (esecuzione via CLI)

> Prima di eseguire: installa `pyyaml` (`pip install pyyaml`) e `dotenv`, poi imposta `NVIDIA_API_KEY` in un file `.env` in questa directory (`export NVIDIA_API_KEY={your-nvidia-api-key}`).


In [1]:
from pathlib import Path
import os, sys, subprocess, shlex
import yaml

CONFIG_PATH = Path("pipeline.yaml")
REPO_ROOT = Path(".").resolve()

print("Repo:", REPO_ROOT)
print("Config:", CONFIG_PATH.resolve())

if not os.path.exists(".env"):
    with open(".env", "w") as f:
        f.writeline("export NVIDIA_API_KEY={your-nvidia-api-key}")


Repo: /home/salvatorecapuozzo/AutoPBR
Config: /home/salvatorecapuozzo/AutoPBR/pipeline.yaml


## Helper: run comandi con output live


In [2]:
from dotenv import load_dotenv
from huggingface_hub import notebook_login

load_dotenv()
notebook_login()

def run(cmd: str):
    print("\n▶", cmd)
    subprocess.run(shlex.split(cmd), check=True)

def q(x) -> str:
    return shlex.quote(str(x))

def args_to_cli(args: dict, ctx: dict) -> str:
    parts = []
    for k, v in args.items():
        flag = f"--{k}"
        if isinstance(v, bool):
            if v:
                parts.append(flag)
            continue
        if isinstance(v, str):
            v = v.format(**ctx)
        parts.append(flag)
        parts.append(str(v))
    return " ".join(q(p) for p in parts)


## Load config + sanity checks


In [3]:
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing {CONFIG_PATH}. Create/commit it in the repo.")

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

photos_dir = cfg["paths"]["photos_dir"]
out_root   = cfg["paths"]["output_root"]
ctx = {"photos_dir": photos_dir, "output_root": out_root}

# Env check
missing = [k for k in cfg.get("env", {}).get("required", []) if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing env vars: {missing}. Set them before running (e.g., export NVIDIA_API_KEY=...).")
print("✅ Env OK")

# Input folder check
photos_path = REPO_ROOT / photos_dir
photos_path.mkdir(exist_ok=True, parents=True)
imgs = list(photos_path.glob("*.jpg")) + list(photos_path.glob("*.jpeg")) + list(photos_path.glob("*.png"))
print(f"📷 images in {photos_path}: {len(imgs)}")


✅ Env OK
📷 images in /home/salvatorecapuozzo/AutoPBR/photos: 1529


## Step 1 — SAM3 (segmentation)

Nota: i prompt nel YAML sono documentativi; se `2_sam3hi.py` non espone flag per i prompt, li ignora comunque.
Qui passiamo solo flag stabili (paths / per_building / bbox gating) *se presenti nel YAML*.


In [19]:
sam3 = cfg.get("sam3", {})
if sam3.get("enabled", True):
    script = sam3.get("script", "2_sam3hi.py")

    print(sam3)

    # pass only known scalar args if provided
    args = {}
    for k in ["out_dir", "score_thr", "mask_thr", "images_dir", "pattern"]:
        if k in sam3:
            kk = k.replace("_", "-")
            args[kk] = sam3[k]

    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ SAM3 disabled in config")


{'enabled': True, 'script': '2_sam3hi.py', 'classes': {'wall': ['building facade', 'wall', 'building front'], 'roof': ['building roof', 'roof'], 'road': ['road', 'street', 'road surface', 'asphalt road', 'driveway', 'sidewalk', 'pavement', 'footpath', 'walkway', 'pedestrian walkway']}, 'per_building': True, 'min_inst_bbox_h_wall': 70, 'min_inst_bbox_h_roof': 110, 'out_dir': '{output_root}/sam3_instances', 'images_dir': '{photos_dir}', 'pattern': '*.jpg', 'score_thr': 0.6, 'mask_thr': 0.5}

▶ /home/salvatorecapuozzo/anaconda3/envs/dl_env/bin/python /home/salvatorecapuozzo/AutoPBR/2_sam3hi.py --out-dir data_output/sam3_instances --score-thr 0.6 --mask-thr 0.5 --images-dir photos --pattern '*.jpg'
🖼️  Found 1529 images in: photos
🔄 Loading SAM3...


Loading weights: 100%|██████████| 1468/1468 [00:01<00:00, 744.04it/s, Materializing param=vision_encoder.neck.fpn_layers.3.proj2.weight]                        


✅ SAM3 ready | device=cuda | dtype=torch.float16
⚙️  score_thr=0.6 | mask_thr=0.5 | road_debug=True

📸 Processing 1003317145127470.jpg
🏢 Found 5 buildings

📸 Processing 1004886326984596.jpg
🏢 Found 1 buildings

📸 Processing 1005755808085218.jpg
🏢 Found 6 buildings

📸 Processing 1006395093438916.jpg
🏢 Found 11 buildings

📸 Processing 1009719100838582.jpg
🏢 Found 20 buildings

📸 Processing 1010188449511786.jpg
🏢 Found 5 buildings

📸 Processing 1010335640916933.jpg
🏢 Found 3 buildings

📸 Processing 1014882470163721.jpg
🏢 Found 11 buildings

📸 Processing 1018594233637353.jpg
🏢 Found 4 buildings

📸 Processing 1018795263283027.jpg
🏢 Found 6 buildings

📸 Processing 101989905547842.jpg
🏢 Found 7 buildings

📸 Processing 102233875481259.jpg
🏢 Found 6 buildings

📸 Processing 1022931445198175.jpg
🏢 Found 8 buildings

📸 Processing 1023239621541664.jpg
🏢 Found 3 buildings

📸 Processing 1024408862351013.jpg
🏢 Found 6 buildings

📸 Processing 1029104787496118.jpg
🏢 Found 16 buildings

📸 Processing 1031

## Step 2 — Proxy semantic check


In [23]:
proxy = cfg.get("proxy_check", {})
if proxy.get("enabled", True):
    script = proxy.get("script", "3_proxy_semantic_check.py")
    args = {k: v for k, v in proxy.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ proxy_check disabled in config")



▶ /home/salvatorecapuozzo/anaconda3/envs/dl_env/bin/python /home/salvatorecapuozzo/AutoPBR/3_proxy_semantic_check.py --out_dir data_output/sam3_instances --manifest data_output/sam3_instances/manifest.json
🔌 device=cuda
🅰️ Proxy A (SegFormer): nvidia/segformer-b5-finetuned-ade-640-640
🅱️ Proxy B (OneFormer): shi-labs/oneformer_ade20k_swin_large


Loading weights: 100%|██████████| 1172/1172 [00:00<00:00, 9469.94it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]             
The image processor of type `OneFormerImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 826/826 [00:00<00:00, 8825.26it/s, Materializing param=model.transformer_module.queries_embedder.weight]                                                       


✅ Done
📄 data_output/sam3_instances/proxy_global_iou.csv
📄 data_output/sam3_instances/proxy_per_building_iou.csv


## Step 3 — Nemotron structured (ECCV frozen config)


In [31]:
ns = cfg.get("nemotron_structured", {})
if ns.get("enabled", True):
    script = ns.get("script", "4_nemotron_building_priors.py")
    args = {k: v for k, v in ns.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx)
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ nemotron_structured disabled in config")



▶ /home/salvatorecapuozzo/anaconda3/envs/dl_env/bin/python /home/salvatorecapuozzo/AutoPBR/4_nemotron_building_priors.py --model nvidia/llama-3.1-nemotron-nano-vl-8b-v1 --canon_mode soft --per_building --skip_far_buildings --manifest data_output/sam3_instances/manifest.json --masked_dir data_output/masked_rgb --out_json materials_v2_filtered.json --min_cov 1.0 --min_cov_roof 1.5 --min_cov_building 0.5 --min_cov_roof_building 0.5 --roof_conf_threshold 0.6 --crop_pad 24 --min_crop_side 64 --max_retries 3 --retry_backoff 0.8
[1/1529] 📸 1003317145127470
   ✓ global road: brick | conf=1.00 (32.2%)
   ✓ global wall: wood | conf=0.80 (44.0%)
   ✓ building_00 wall: wood | conf=0.80 (21.5%)
   ✓ building_01 wall: wood | conf=0.80 (21.1%)
[2/1529] 📸 1004886326984596
   ✓ global road: concrete | conf=0.90 (15.1%)
   ✓ global wall: stone | conf=0.80 (40.5%)
   ✓ building_00 wall: stone | conf=0.80 (38.5%)
[3/1529] 📸 1005755808085218
   ✓ global road: asphalt | conf=0.90 (42.2%)
   ✓ global wall: 

## Step 4 — Baseline full-image (optional)


In [4]:
nbaseline = cfg.get("nemotron_baseline", {})
if nbaseline.get("enabled", False):
    script = nbaseline.get("script", "5_nemotron_fullimage.py")
    args = {k: v for k, v in nbaseline.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx)
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ baseline disabled in config")



▶ /home/salvatorecapuozzo/anaconda3/envs/dl_env/bin/python /home/salvatorecapuozzo/AutoPBR/5_nemotron_fullimage.py --model nvidia/llama-3.1-nemotron-nano-vl-8b-v1 --prompt_type two_pass_json_review --source_manifest data_output/sam3_instances/manifest.json --output_json baseline_full_image.json
[1/1529] 📸 image_1
   road: brick (conf=1.00)
   wall: wood (conf=1.00)
   roof: other:not_visible (conf=1.00)
[2/1529] 📸 image_2
   road: stone (conf=1.00)
   wall: stone (conf=1.00)
   roof: other:not_visible (conf=1.00)
[3/1529] 📸 image_3
   road: concrete (conf=0.90)
   wall: concrete (conf=0.80)
   roof: metal (conf=0.70)
[4/1529] 📸 image_4
   road: asphalt (conf=0.80)
   wall: other:not_visible (conf=1.00)
   roof: other:not_visible (conf=1.00)
[5/1529] 📸 image_5
   road: asphalt (conf=1.00)
   wall: other:not_visible (conf=0.00)
   roof: other:not_visible (conf=0.00)
[6/1529] 📸 image_6
   road: asphalt (conf=1.00)
   wall: other:not_visible (conf=1.00)
   roof: other:not_visible (conf=1.

Traceback (most recent call last):
  File "/home/salvatorecapuozzo/AutoPBR/5_nemotron_fullimage.py", line 330, in <module>
    main()
  File "/home/salvatorecapuozzo/AutoPBR/5_nemotron_fullimage.py", line 287, in main
    raw_pred, canon_pred = analyze_full_image(
                           ^^^^^^^^^^^^^^^^^^^
  File "/home/salvatorecapuozzo/AutoPBR/5_nemotron_fullimage.py", line 185, in analyze_full_image
    resp = client.chat.completions.create(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/salvatorecapuozzo/anaconda3/envs/dl_env/lib/python3.12/site-packages/openai/_utils/_utils.py", line 286, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/salvatorecapuozzo/anaconda3/envs/dl_env/lib/python3.12/site-packages/openai/resources/chat/completions/completions.py", line 1192, in create
    return self._post(
           ^^^^^^^^^^^
  File "/home/salvatorecapuozzo/anaconda3/envs/dl_env/lib/python3.12/site-packages/openai/_base_client.py",

KeyboardInterrupt: 

## Step 5 — Validation (optional)


In [ ]:
val = cfg.get("validation", {})
if val.get("enabled", False):
    script = val.get("script", "validator.py")
    args = {k: v for k, v in val.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ validation disabled in config")


## Quick check outputs


In [ ]:
out_candidates = [
    REPO_ROOT / "materials_v2_filtered.json",
    REPO_ROOT / "baseline_full_image.json",
    REPO_ROOT / out_root / "sam3_instances" / "manifest.json",
    REPO_ROOT / out_root / "validation_results" / "report.csv",
]
for p in out_candidates:
    print(("✅" if p.exists() else "❌"), p)
